In [2]:
import pandas as pd

In [19]:
db=pd.read_csv(r"C:\Users\Gabo\Downloads\metadata_level_pooling_level.csv", index_col=0)

In [25]:
def getbest(db:pd.DataFrame,groupcols: list,metric_cols:list, loss_columns:list=['val_loss','train_loss']):
    db1=db.copy()
    agregaciones={
        f"{col}":(col ,'mean')
        for col in metric_cols
    }
    tolenrance=5e-2
    db2=db1[db1['epoch']==db1['best_epoch']].copy()
    #db2=db1.copy()
    db3=db2.groupby(by=groupcols)[metric_cols].agg(**agregaciones).reset_index()
    db3['no_overfiting']=db3.apply(
        lambda x : (abs(x[metric_cols[0]]-x[metric_cols[1]])<tolenrance ) or
        (abs(x[metric_cols[2]]-x[metric_cols[3]])<tolenrance )
    , axis=1)
    db4=db3[db3['no_overfiting']]
    result=dict(db4.sort_values(metric_cols[:2], ascending=False).iloc[0])
    
    
    return result, db3
    
    

In [24]:
5e-2

0.05

In [26]:
db.columns

Index(['epoch', 'train_loss', 'train_accuracy', 'train_f1', 'val_loss',
       'val_accuracy', 'val_f1', 'best_epoch', 'pooling', 'input_dim',
       'hidden_dim', 'num_hidden_layers', 'activation', 'normalization',
       'dropout', 'device', 'seed'],
      dtype='object')

In [27]:
groupcols=['pooling']
metrics= ['train_f1','val_f1','train_accuracy','val_accuracy']
t1, dbt=getbest(db,groupcols,metrics)


In [28]:
dbt

,pooling,train_f1,val_f1,train_accuracy,val_accuracy,no_overfiting
0,cls,0.760182,0.749285,0.767998,0.771917,True
1,max,0.886490,0.852861,0.890974,0.870019,True
2,mean,0.948142,0.925263,0.950565,0.929442,True


In [6]:
t1

{'pooling': 'cls',
 'train_f1': np.float64(0.7601815405195581),
 'val_f1': np.float64(0.7492849386487876),
 'train_accuracy': np.float64(0.7679979904546596),
 'val_accuracy': np.float64(0.7719174871073605),
 'no_overfiting': np.True_}

In [49]:
from itertools import product
num_hidden_layers_options = [0, 1, 2, 3]
hidden_dim_options = [128, 256, 512]
activation_options = [ "gelu",    "relu",    "silu"]
normalization_options = [    None,    "layernorm",    "batchnorm"]
dropout_options = [    0.0,    0.1,    0.3,    0.5]


In [67]:
print(27*16/8)

54.0


In [63]:
a=list(product(num_hidden_layers_options,hidden_dim_options,
               activation_options,normalization_options,
               dropout_options))
config = [ "num_hidden_layers",
                "hidden_dim",
                "activation",
                "normalization",
                "dropout"]
configs=list(map(
    lambda x: dict(zip(config,x))
    ,a))





In [34]:
t1.sort_values(['train_f1','val_f1'], ascending=False).iloc[0]

pooling               mean
hidden_dim             512
train_f1          0.998244
val_f1            0.996878
train_accuracy    0.997189
val_accuracy       0.99499
no_overfiting         True
Name: 8, dtype: object

In [29]:
a=abs(t1.iloc[4,2]-t1.iloc[4,3])
print(t1.iloc[4,3]+a)

0.990220412489757
